In [7]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

In [8]:
#   import warnings
#   
#   warnings.filterwarnings("ignore")

In [9]:
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

print("Project dir:", PROJECT_DIR.name)
print("Working dir:", cwd.name)

Project dir: 2026-03_taxi-trip-chicago
Working dir: notebooks


In [ ]:
DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_STAGING = PROJECT_DIR / "data" / "staging"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"

TRIPS_FILE = DATA_RAW / "Taxi_Trips_20260324.csv"

In [11]:
df = pd.read_csv(TRIPS_FILE)
print("trips shape:", df.shape)
df.head()

trips shape: (14219363, 23)


,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,0000184e7cd53cee95af32eba49c44e4d20adcd8,f538e6b729d1aaad4230e9dcd9dc2fd9a168826ddadbd6...,01/19/2024 05:00:00 PM,01/19/2024 06:00:00 PM,4051.0,17.12,1.703198e+10,1.703132e+10,76.0,32.0,...,4.0,60.00,Credit Card,Flash Cab,41.979071,-87.903040,POINT (-87.9030396611 41.9790708201),41.884987,-87.620993,POINT (-87.6209929134 41.8849871918)
1,000072ee076c9038868e239ca54185eb43959db0,e51e2c30caec952b40b8329a68b498e18ce8a1f40fa75c...,01/28/2024 02:30:00 PM,01/28/2024 03:00:00 PM,1749.0,12.70,NaN,NaN,6.0,NaN,...,0.0,33.75,Cash,Flash Cab,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),NaN,NaN,NaN
2,000074019d598c2b1d6e77fbae79e40b0461a2fc,aeb280ef3be3e27e081eb6e76027615b0d40925b84d3eb...,01/05/2024 09:00:00 AM,01/05/2024 09:00:00 AM,517.0,3.39,NaN,NaN,6.0,8.0,...,1.0,14.69,Mobile,Taxicab Insurance Agency Llc,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),41.899602,-87.633308,POINT (-87.6333080367 41.899602111)
3,00007572c5f92e2ff067e6f838a5ad74e83665d3,7d21c2ca227db8f27dda96612bfe5520ab408fa9a462c8...,01/22/2024 08:45:00 AM,01/22/2024 09:30:00 AM,2050.0,15.06,NaN,NaN,76.0,NaN,...,5.5,56.56,Credit Card,Globe Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),NaN,NaN,NaN
4,00007c3e7546e2c7d15168586943a9c22c3856cf,8ef1056519939d511d24008e394f83e925d2539d668a00...,01/18/2024 07:15:00 PM,01/18/2024 07:30:00 PM,1004.0,1.18,1.703184e+10,1.703184e+10,32.0,32.0,...,0.0,19.66,Mobile,5 Star Taxi,41.880994,-87.632746,POINT (-87.6327464887 41.8809944707),41.880994,-87.632746,POINT (-87.6327464887 41.8809944707)


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14219363 entries, 0 to 14219362
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     object 
 1   Taxi ID                     object 
 2   Trip Start Timestamp        object 
 3   Trip End Timestamp          object 
 4   Trip Seconds                float64
 5   Trip Miles                  float64
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Fare                        float64
 11  Tips                        float64
 12  Tolls                       float64
 13  Extras                      float64
 14  Trip Total                  float64
 15  Payment Type                object 
 16  Company                     object 
 17  Pickup Centroid Latitude    float64
 18  Pickup Centroid Longitude   float64
 19  Pickup Centroid Loc

quito espacios de nombres de columnas

In [13]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
)

In [14]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid__location'],
      dtype='object')

dropoff_centroid_location viene con dos guines bajos, se modifica puntual ese nombre

In [15]:
df = df.rename(columns={
    "dropoff_centroid__location": "dropoff_centroid_location"
})

formato a fechas

In [16]:
df["trip_start_timestamp"] = pd.to_datetime(
    df["trip_start_timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

df["trip_end_timestamp"] = pd.to_datetime(
    df["trip_end_timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

In [17]:
df[["trip_start_timestamp", "trip_end_timestamp"]].head()

,trip_start_timestamp,trip_end_timestamp
0,2024-01-19 17:00:00,2024-01-19 18:00:00
1,2024-01-28 14:30:00,2024-01-28 15:00:00
2,2024-01-05 09:00:00,2024-01-05 09:00:00
3,2024-01-22 08:45:00,2024-01-22 09:30:00
4,2024-01-18 19:15:00,2024-01-18 19:30:00


In [18]:
cols_num = [
    "trip_seconds",
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "pickup_centroid_latitude",
    "pickup_centroid_longitude",
    "dropoff_centroid_latitude",
    "dropoff_centroid_longitude"
]

for c in cols_num:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [19]:
cols_id_geo = [
    "pickup_census_tract",
    "dropoff_census_tract",
    "pickup_community_area",
    "dropoff_community_area"
]

for c in cols_id_geo:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14219363 entries, 0 to 14219362
Data columns (total 23 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   trip_id                     object        
 1   taxi_id                     object        
 2   trip_start_timestamp        datetime64[ns]
 3   trip_end_timestamp          datetime64[ns]
 4   trip_seconds                float64       
 5   trip_miles                  float64       
 6   pickup_census_tract         float64       
 7   dropoff_census_tract        float64       
 8   pickup_community_area       float64       
 9   dropoff_community_area      float64       
 10  fare                        float64       
 11  tips                        float64       
 12  tolls                       float64       
 13  extras                      float64       
 14  trip_total                  float64       
 15  payment_type                object        
 16  company         

In [21]:
df.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location
0,0000184e7cd53cee95af32eba49c44e4d20adcd8,f538e6b729d1aaad4230e9dcd9dc2fd9a168826ddadbd6...,2024-01-19 17:00:00,2024-01-19 18:00:00,4051.0,17.12,1.703198e+10,1.703132e+10,76.0,32.0,...,4.0,60.00,Credit Card,Flash Cab,41.979071,-87.903040,POINT (-87.9030396611 41.9790708201),41.884987,-87.620993,POINT (-87.6209929134 41.8849871918)
1,000072ee076c9038868e239ca54185eb43959db0,e51e2c30caec952b40b8329a68b498e18ce8a1f40fa75c...,2024-01-28 14:30:00,2024-01-28 15:00:00,1749.0,12.70,NaN,NaN,6.0,NaN,...,0.0,33.75,Cash,Flash Cab,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),NaN,NaN,NaN
2,000074019d598c2b1d6e77fbae79e40b0461a2fc,aeb280ef3be3e27e081eb6e76027615b0d40925b84d3eb...,2024-01-05 09:00:00,2024-01-05 09:00:00,517.0,3.39,NaN,NaN,6.0,8.0,...,1.0,14.69,Mobile,Taxicab Insurance Agency Llc,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),41.899602,-87.633308,POINT (-87.6333080367 41.899602111)
3,00007572c5f92e2ff067e6f838a5ad74e83665d3,7d21c2ca227db8f27dda96612bfe5520ab408fa9a462c8...,2024-01-22 08:45:00,2024-01-22 09:30:00,2050.0,15.06,NaN,NaN,76.0,NaN,...,5.5,56.56,Credit Card,Globe Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),NaN,NaN,NaN
4,00007c3e7546e2c7d15168586943a9c22c3856cf,8ef1056519939d511d24008e394f83e925d2539d668a00...,2024-01-18 19:15:00,2024-01-18 19:30:00,1004.0,1.18,1.703184e+10,1.703184e+10,32.0,32.0,...,0.0,19.66,Mobile,5 Star Taxi,41.880994,-87.632746,POINT (-87.6327464887 41.8809944707),41.880994,-87.632746,POINT (-87.6327464887 41.8809944707)


In [22]:
df.isna().sum().sort_values(ascending=False)

dropoff_census_tract          8113518
pickup_census_tract           7922844
dropoff_community_area        1272573
dropoff_centroid_longitude    1196124
dropoff_centroid_location     1196124
dropoff_centroid_latitude     1196124
pickup_community_area          397311
pickup_centroid_location       390023
pickup_centroid_latitude       390023
pickup_centroid_longitude      390023
trip_total                      30499
fare                            30499
extras                          30499
tolls                           30499
tips                            30499
trip_seconds                     2649
trip_end_timestamp                189
trip_miles                        119
taxi_id                            11
trip_id                             0
trip_start_timestamp                0
payment_type                        0
company                             0
dtype: int64

In [23]:
(df.isna().mean() * 100).sort_values(ascending=False).round(2)

dropoff_census_tract          57.06
pickup_census_tract           55.72
dropoff_community_area         8.95
dropoff_centroid_longitude     8.41
dropoff_centroid_location      8.41
dropoff_centroid_latitude      8.41
pickup_community_area          2.79
pickup_centroid_location       2.74
pickup_centroid_latitude       2.74
pickup_centroid_longitude      2.74
trip_total                     0.21
fare                           0.21
extras                         0.21
tolls                          0.21
tips                           0.21
trip_seconds                   0.02
trip_end_timestamp             0.00
trip_miles                     0.00
taxi_id                        0.00
trip_id                        0.00
trip_start_timestamp           0.00
payment_type                   0.00
company                        0.00
dtype: float64

se crean nuevas variables 

In [24]:
df["trip_date"] = df["trip_start_timestamp"].dt.date
df["trip_year"] = df["trip_start_timestamp"].dt.year
df["trip_month"] = df["trip_start_timestamp"].dt.month
df["trip_day"] = df["trip_start_timestamp"].dt.day
df["trip_hour"] = df["trip_start_timestamp"].dt.hour
df["trip_weekday"] = df["trip_start_timestamp"].dt.day_name()
df["trip_weekday_num"] = df["trip_start_timestamp"].dt.weekday

df["is_weekend"] = df["trip_weekday_num"].isin([5, 6])

In [25]:
df[["trip_start_timestamp", "trip_date", "trip_year", "trip_month", "trip_hour", "trip_weekday", "trip_weekday_num"]].head()

,trip_start_timestamp,trip_date,trip_year,trip_month,trip_hour,trip_weekday,trip_weekday_num
0,2024-01-19 17:00:00,2024-01-19,2024,1,17,Friday,4
1,2024-01-28 14:30:00,2024-01-28,2024,1,14,Sunday,6
2,2024-01-05 09:00:00,2024-01-05,2024,1,9,Friday,4
3,2024-01-22 08:45:00,2024-01-22,2024,1,8,Monday,0
4,2024-01-18 19:15:00,2024-01-18,2024,1,19,Thursday,3


In [26]:
df[["trip_seconds", "trip_miles", "fare", "tips", "tolls", "extras", "trip_total"]].describe()

,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total
count,1.421671e+07,1.421924e+07,1.418886e+07,1.418886e+07,1.418886e+07,1.418886e+07,1.418886e+07
mean,1.225905e+03,6.567188e+00,2.246990e+01,2.770886e+00,3.059485e-02,2.033563e+00,2.754610e+01
std,1.610112e+03,7.600035e+00,3.317939e+01,4.236935e+00,3.815459e+00,8.786831e+00,3.744949e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.800000e+02,1.070000e+00,8.500000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.025000e+01
50%,9.000000e+02,3.060000e+00,1.500000e+01,0.000000e+00,0.000000e+00,0.000000e+00,1.778000e+01
75%,1.663000e+03,1.165000e+01,3.375000e+01,4.000000e+00,0.000000e+00,1.500000e+00,4.120000e+01
max,8.640000e+04,3.397800e+03,9.999750e+03,7.507500e+02,5.550000e+03,5.559500e+03,9.999750e+03


In [27]:
df["trip_start_timestamp"].min(), df["trip_start_timestamp"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2026-03-01 00:00:00'))

In [28]:
df["flag_monto"] = df["fare"].notna()

df["flag_pickup_geo"] = (
    df["pickup_community_area"].notna() |
    df["pickup_centroid_latitude"].notna() |
    df["pickup_centroid_longitude"].notna()
)

df["flag_dropoff_geo"] = (
    df["dropoff_community_area"].notna() |
    df["dropoff_centroid_latitude"].notna() |
    df["dropoff_centroid_longitude"].notna()
)

In [29]:
df[["flag_monto", "flag_pickup_geo", "flag_dropoff_geo"]].head()

,flag_monto,flag_pickup_geo,flag_dropoff_geo
0,True,True,True
1,True,True,False
2,True,True,True
3,True,True,False
4,True,True,True


In [30]:
df["trip_minutes"] = df["trip_seconds"] / 60
df["trip_hours"] = df["trip_seconds"] / 3600

In [31]:
df["total_calculado"] = df["fare"] + df["tips"] + df["tolls"] + df["extras"]

In [33]:
#   df["diff_total"] = df["trip_total"] - df["total_calculado"]

In [34]:
# df["flag_total_consistente"] = df["diff_total"].abs() <= 0.01

In [35]:
# df["flag_total_consistente"].mean()

In [36]:
df["speed_mph"] = df["trip_miles"] / df["trip_hours"]

In [37]:
df["fare_per_mile"] = df["fare"] / df["trip_miles"]

In [38]:
df["trip_total_per_mile"] = df["trip_total"] / df["trip_miles"]

In [39]:
df["fare_per_minute"] = df["fare"] / df["trip_minutes"]

In [40]:
df["trip_total_per_minute"] = df["trip_total"] / df["trip_minutes"]

In [41]:
import numpy as np

cols_ratio = [
    "speed_mph",
    "fare_per_mile",
    "trip_total_per_mile",
    "fare_per_minute",
    "trip_total_per_minute"
]

for c in cols_ratio:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan)

In [42]:
df["tip_pct"] = (df["tips"] / df["fare"]) * 100
df["tip_pct"] = df["tip_pct"].replace([np.inf, -np.inf], np.nan)

el viaje tiene el núcleo mínimo para análisis operativo:

inicio + duración + distancia

In [43]:
df["flag_trip_core"] = (
    df["trip_seconds"].notna() &
    df["trip_miles"].notna() &
    df["trip_start_timestamp"].notna()
)

In [44]:
df["time_band"] = "night"

df.loc[df["trip_hour"].between(6, 11), "time_band"] = "morning"
df.loc[df["trip_hour"].between(12, 17), "time_band"] = "afternoon"
df.loc[df["trip_hour"].between(18, 23), "time_band"] = "evening"

In [46]:
df[
    [
        "trip_year", "trip_month", "trip_hour", "is_weekend", "time_band",
        "trip_minutes", "trip_hours", "total_calculado", "speed_mph", "fare_per_mile",
        "trip_total_per_mile", "fare_per_minute", "trip_total_per_minute",
        "tip_pct", "flag_trip_core"
    ]
].head()

,trip_year,trip_month,trip_hour,is_weekend,time_band,trip_minutes,trip_hours,total_calculado,speed_mph,fare_per_mile,trip_total_per_mile,fare_per_minute,trip_total_per_minute,tip_pct,flag_trip_core
0,2024,1,17,False,afternoon,67.516667,1.125278,59.50,15.214021,2.657710,3.504673,0.673908,0.888669,21.978022,True
1,2024,1,14,True,afternoon,29.150000,0.485833,33.75,26.140652,2.657480,2.657480,1.157804,1.157804,0.000000,True
2,2024,1,9,False,morning,8.616667,0.143611,14.69,23.605416,3.218289,4.333333,1.266151,1.704836,25.481210,True
3,2024,1,8,False,morning,34.166667,0.569444,56.06,26.446829,2.606242,3.755644,1.148780,1.655415,28.815287,True
4,2024,1,19,False,evening,16.733333,0.278889,19.66,4.231076,13.508475,16.661017,0.952590,1.174900,23.337516,True


In [48]:
df[
    [
        "trip_minutes", "trip_hours", "speed_mph",
        "fare_per_mile", "trip_total_per_mile", "fare_per_minute",
        "trip_total_per_minute", "tip_pct"
    ]
].describe()

,trip_minutes,trip_hours,speed_mph,fare_per_mile,trip_total_per_mile,fare_per_minute,trip_total_per_minute,tip_pct
count,1.421671e+07,1.421671e+07,1.402146e+07,1.293828e+07,1.293828e+07,1.399218e+07,1.399218e+07,1.415224e+07
mean,2.043176e+01,3.405293e-01,1.759367e+01,1.376095e+01,1.778216e+01,1.215641e+01,1.459689e+01,1.379367e+01
std,2.683520e+01,4.472533e-01,1.747083e+02,1.774354e+02,2.224980e+02,2.122776e+02,2.191977e+02,1.666939e+02
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.000000e+00,1.333333e-01,8.400000e+00,2.685811e+00,3.300000e+00,8.230453e-01,9.508887e-01,0.000000e+00
50%,1.500000e+01,2.500000e-01,1.338215e+01,3.688525e+00,4.444444e+00,1.041667e+00,1.241026e+00,0.000000e+00
75%,2.771667e+01,4.619444e-01,2.395030e+01,5.973451e+00,7.148936e+00,1.393189e+00,1.723529e+00,2.274809e+01
max,1.440000e+03,2.400000e+01,4.233044e+05,1.020500e+05,1.020500e+05,2.265000e+05,2.265000e+05,2.650000e+05


In [49]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid_location', 'trip_date', 'trip_year', 'trip_month',
       'trip_day', 'trip_hour', 'trip_weekday', 'trip_weekday_num',
       'is_weekend', 'flag_monto', 'flag_pickup_geo', 'flag_dropoff_geo',
       'trip_minutes', 'trip_hours', 'total_calculado', 'speed_mph',
       'fare_per_mile', 'trip_total_per_mile', 'fare_per_minute',
       'trip_total_per_minute', 'tip_pct', 'flag_trip_core', 'time_band'],
      dtype='object')

In [50]:
df.loc[df["trip_hours"] <= 0, "speed_mph"] = np.nan

df.loc[df["trip_miles"] <= 0, "fare_per_mile"] = np.nan
df.loc[df["trip_miles"] <= 0, "trip_total_per_mile"] = np.nan

df.loc[df["trip_minutes"] <= 0, "fare_per_minute"] = np.nan
df.loc[df["trip_minutes"] <= 0, "trip_total_per_minute"] = np.nan

df.loc[df["fare"] <= 0, "tip_pct"] = np.nan

análisis de duplicados

In [51]:
df["trip_id"].duplicated().sum()

np.int64(0)

In [52]:
df.duplicated(subset=["taxi_id", "trip_start_timestamp", "trip_end_timestamp"]).sum()

np.int64(170793)

In [53]:
cols = [
    "trip_seconds",
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "trip_minutes",
    "trip_hours",
    "speed_mph",
    "fare_per_mile",
    "trip_total_per_mile",
    "fare_per_minute",
    "trip_total_per_minute",
    "tip_pct"
]

df[cols].agg(["min", "max"]).T

,min,max
trip_seconds,0.0,86400.000000
trip_miles,0.0,3397.800000
fare,0.0,9999.750000
tips,0.0,750.750000
tolls,0.0,5550.000000
extras,0.0,5559.500000
trip_total,0.0,9999.750000
trip_minutes,0.0,1440.000000
trip_hours,0.0,24.000000
speed_mph,0.0,423304.363636


In [54]:
df[cols].describe(percentiles=[0.01, 0.02, 0.05, 0.50, 0.95, 0.98, 0.99]).round(2).T

,count,mean,std,min,1%,2%,5%,50%,95%,98%,99%,max
trip_seconds,14216714.0,1225.91,1610.11,0.0,0.00,4.00,27.00,900.00,3195.00,3965.00,4581.00,86400.00
trip_miles,14219244.0,6.57,7.60,0.0,0.00,0.00,0.00,3.06,18.32,22.09,26.77,3397.80
fare,14188864.0,22.47,33.18,0.0,3.25,3.25,4.50,15.00,52.25,65.25,75.00,9999.75
tips,14188864.0,2.77,4.24,0.0,0.00,0.00,0.00,0.00,11.10,13.90,16.00,750.75
tolls,14188864.0,0.03,3.82,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,5550.00
extras,14188864.0,2.03,8.79,0.0,0.00,0.00,0.00,0.00,6.00,24.00,32.29,5559.50
trip_total,14188864.0,27.55,37.45,0.0,3.25,3.25,5.04,17.78,68.25,85.00,99.90,9999.75
trip_minutes,14216714.0,20.43,26.84,0.0,0.00,0.07,0.45,15.00,53.25,66.08,76.35,1440.00
trip_hours,14216714.0,0.34,0.45,0.0,0.00,0.00,0.01,0.25,0.89,1.10,1.27,24.00
speed_mph,14021457.0,17.59,174.71,0.0,0.00,0.00,0.00,13.38,40.67,46.88,50.60,423304.36


In [55]:
(df["trip_miles"] == 0).sum()

np.int64(1253227)

In [56]:
(df["fare"] == 0).sum()
(df["trip_total"] == 0).sum()

np.int64(34754)

In [57]:
df.sort_values("trip_miles", ascending=False)[
    ["trip_id", "trip_start_timestamp", "trip_seconds", "trip_miles", "fare", "trip_total", "payment_type", "company"]
].head(20)

,trip_id,trip_start_timestamp,trip_seconds,trip_miles,fare,trip_total,payment_type,company
1727058,8bb9ae67ccafb402045fd52e5f2ba31ef2963835,2024-04-21 07:15:00,8092.0,3397.80,67.00,67.00,Cash,Chicago Independents
673741,8656832cf4c4d23c8bbb728f43bd645020b1258f,2024-02-10 10:00:00,27853.0,3093.47,9999.75,9999.75,Cash,Blue Ribbon Taxi Association
2853227,80e84d4e2ee8e03cf26c994714700b6a7eacf4c6,2024-06-25 14:00:00,2313.0,3017.61,44.25,58.50,Credit Card,Sun Taxi
6373816,c0198d5377db37f1508aa57c8a299d18f2c5b566,2024-12-01 09:30:00,14925.0,2949.41,9999.75,9999.75,Cash,Blue Ribbon Taxi Association
2119521,522fc9a30df54459fbee9558749bbd29674ae700,2024-05-28 16:00:00,72.0,2820.67,3.50,6.00,Cash,Flash Cab
5612768,45c7b37478c4cd41b3abe89bbcf390a8eb2fc9ab,2024-11-25 13:45:00,1732.0,2537.51,42.75,56.70,Credit Card,Choice Taxi Association Inc
3616284,65d80b26805e9548766a10a894848f882374660b,2024-08-06 08:30:00,33950.0,2265.43,9999.75,9999.75,Cash,Blue Ribbon Taxi Association
6151458,52c4902403064408e66026dfa323bb13925b5eb7,2024-12-24 00:15:00,1585.0,2166.39,30.00,30.00,Prcard,5 Star Taxi
188038,6e4aecb7dab42a2109ec97a76e408699e2ad0c62,2024-01-31 12:45:00,56.0,1626.85,3668.50,3668.50,Cash,Blue Ribbon Taxi Association
4442294,483cc8023cff3ac64bc40745d35974cd8c99ab92,2024-09-09 10:30:00,11.0,1293.43,5938.00,5938.00,Cash,Blue Ribbon Taxi Association


In [58]:
df.sort_values("trip_seconds", ascending=False)[
    ["trip_id", "trip_start_timestamp", "trip_seconds", "trip_miles", "fare", "trip_total", "payment_type", "company"]
].head(20)

,trip_id,trip_start_timestamp,trip_seconds,trip_miles,fare,trip_total,payment_type,company
10395891,0cb6338958a53bc3d1140d6cbfa24533a171c897,2025-08-20 15:00:00,86400.0,0.00,1.00,1.00,Cash,Tac - Checker Cab Dispatch
12164258,ec6c7c71e365fc935278e4e5f0c1f88adbfe9a5f,2025-10-05 17:30:00,86396.0,0.00,25.55,25.55,Cash,Chicago Independents
8123455,5224d3ab360fc1ba9a660f92b7c471bc0383818d,2025-03-19 18:15:00,86396.0,1.47,9.25,10.25,Cash,Taxicab Insurance Agency Llc
1495930,1e10bbfb70b9ac63d8c37f191b8760eae77b4399,2024-04-24 22:15:00,86379.0,15.34,38.75,42.75,Cash,Chicago Independents
1126849,7050303945c2d4a577581455c6e9879bd727a8a0,2024-03-18 12:30:00,86340.0,0.00,49.25,49.75,Credit Card,Globe Taxi
499735,23d18b0c751016ef59fb37394155bc556bd5de59,2024-02-19 16:15:00,86340.0,0.00,40.00,50.62,Credit Card,Globe Taxi
2501090,f2172b25cc39040d56d79d5c2376024acdb4acb2,2024-05-07 23:15:00,86340.0,0.00,10.00,10.50,Credit Card,Globe Taxi
5704468,6fc043c9bf81448058a3df76fb89837db9f6ac31,2024-11-13 21:00:00,86322.0,18.51,47.75,55.75,Cash,Flash Cab
3030978,c8c44e4d09621abefef0d5fbd9ee3d13c7d6515d,2024-06-20 21:00:00,86265.0,10.48,32.25,32.25,Cash,Globe Taxi
12150895,e7278bf9671255e1c70ac89445304cf8a6455e18,2025-10-29 16:45:00,86177.0,0.00,13.00,13.00,Cash,Tac - Yellow Cab Association


In [61]:
dup_taxi_time = df[
    df.duplicated(
        subset=["taxi_id", "trip_start_timestamp", "trip_end_timestamp"],
        keep=False
    )
].sort_values(["taxi_id", "trip_start_timestamp", "trip_end_timestamp"])

dup_taxi_time[
    [
        "trip_id", "taxi_id", "trip_start_timestamp", "trip_end_timestamp",
        "trip_seconds", "trip_miles", "fare", "trip_total",
        "payment_type", "company"
    ]
].head(30)

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,fare,trip_total,payment_type,company
205641,789df9ed9d6a2139febdc7ee3784e8eb219ced18,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-01-11 13:45:00,2024-01-11 13:45:00,0.0,0.00,3.25,3.25,Cash,Taxi Affiliation Services
209880,7b1fb96fd071d6af1151437c6f1ae5ec959c5683,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-01-11 13:45:00,2024-01-11 13:45:00,60.0,0.00,4.75,4.75,Cash,Taxi Affiliation Services
312991,b7ae5ea228d6ac282ba17d1310cd625b6ba16d73,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-01-16 14:15:00,2024-01-16 14:15:00,0.0,0.00,3.25,3.25,Cash,Taxi Affiliation Services
414149,f2c96c504f553485b1b51a31c91fe503e7c0f0c9,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-01-16 14:15:00,2024-01-16 14:15:00,240.0,0.00,5.00,5.00,Unknown,Taxi Affiliation Services
70564,294b58c308310f532e2a0df5668d8656718213d3,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-01-19 17:15:00,2024-01-19 17:15:00,360.0,0.00,5.75,7.75,Credit Card,Taxi Affiliation Services
434358,fe9aa56aefb36b60d9cbf85b668492237f0c17ae,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-01-19 17:15:00,2024-01-19 17:15:00,300.0,0.00,5.75,9.75,Credit Card,Taxi Affiliation Services
719817,a08e518dbbfc3eded542ba89f80b2dfa91bdeaa6,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-02-20 14:00:00,2024-02-20 14:00:00,0.0,0.00,3.25,3.25,Cash,Taxi Affiliation Services
869042,f4e72fe120d8bd230a45f31720e2d601898634e5,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-02-20 14:00:00,2024-02-20 14:00:00,180.0,0.00,4.25,4.25,Cash,Taxi Affiliation Services
459098,0cd3c0a0b6bf640ff934bf8f562450a96b805ec5,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-02-28 19:00:00,2024-02-28 19:00:00,0.0,0.00,7.50,9.50,Credit Card,Taxi Affiliation Services
717995,9f8ff91834d5b9130aeb778c7919531ac6ae6e98,0041f8f0c91881c1e1913f2548522495fe3c4c719aa67f...,2024-02-28 19:00:00,2024-02-28 19:00:00,0.0,0.00,3.25,5.25,Cash,Taxi Affiliation Services


In [62]:
dup_taxi_time.groupby(
    ["taxi_id", "trip_start_timestamp", "trip_end_timestamp"]
).size().sort_values(ascending=False).head(20)

taxi_id                                                                                                                           trip_start_timestamp  trip_end_timestamp 
02ef8f01232b1b1828f4e5e1b8e8a85cd71b67c449afafbd2dd7ab726b2bb72795fcd00a9bb2477dfa6f8a325bbfde41ae888127ecd354096a1117f5e6fd9e0d  2024-12-30 01:15:00   2024-12-30 01:15:00    37
1faf8b812afc33b1e1a4f600e67a5a33f13ee8dd8ebf75922e70ffa1085bb6670ed097a30f9b15004f88b4fbee1efc9c029ffdd0d8712e0409469b7c799e433b  2026-02-14 12:45:00   2026-02-14 12:45:00    23
c8f57a1150c210a9e6b3fcfb24c3d6d0a43d1879b4b9795d12ffb5917f2b77c86a098c8f0a7c2a301ef4ea972985d4e05bd8d52a2f7a1ecca253fb8e81e5755e  2024-10-28 10:15:00   2024-10-28 10:15:00    15
222a0a55fe5180e292a8e128804e081fa50b05a0329775f9344b50ee230f23d12635ee6a2429af764d01221057af04260089849cfb0192fa29bfda38362ffad6  2024-03-09 10:45:00   2024-03-09 10:45:00    14
                                                                                                                    

In [63]:
(df["trip_seconds"] == 0).sum()


np.int64(195154)

EXPORT PARQUET para total y CSV para 2026

In [ ]:
#   df.to_parquet(DATA_STAGING / "taxi_trips_chicago_staging.parquet", index=False)

In [66]:
#   # ME QUEDO CON EL 2026 PARA TRAGBAJAR EN DASHBOARDS
#   df_2026 = df[df["trip_year"] == 2026].copy()
#   
#   df_2026.to_csv(DATA_STAGING / "taxi_trips_chicago_2026.csv", index=False)